## 1.3 状态定义
官方推荐的三种方式： TypeDict, dataclass, Pydantic

### 1.3.1 TypeDict(字典状态)
本身是TypedDict的子类，用于定义状态的类型。使用参01_graph.ipynb


### 1.3.2 dataclass

In [1]:
from dataclasses import dataclass
from operator import add
from typing import Annotated

from langgraph.graph import StateGraph, START, END


# graph api风格
# 1、定义状态
@dataclass
class OverAllState:
    # 日志类型还是 list[str], 但更新的方式不是覆盖 而是add追加
    logs: Annotated[list[str], add]
    # 当前运行的位置。当前的集合
    cur_id: str


# 2、定义节点：本质是定义一个执行的流程
# state: OverAllState 输入类型
# -> OverAllState 返回类型
# 当graph 执行到node1 会在日志添加内容，cur_id 拼接上当前id
def node_1(state: OverAllState) -> OverAllState:
    # 获取当前id。当前状态定义为类，需要通过属性获取
    pre_id = state.cur_id
    # python支持 字段与dataclass 字段的映射转换
    return {
        # 拼接上node1
        "cur_id": pre_id + ", node_1",
        # 日志
        "logs": ["node_1 运行完毕"],
    }


def node_2(state: OverAllState) -> OverAllState:
    # 获取当前id
    pre_id = state.cur_id
    # return {
    #     # 拼接上node1
    #     "cur_id": pre_id + ", node_2",
    #     # 日志
    #     "logs": ["node_2 运行完毕"],
    # }
    return OverAllState(
        logs=state.logs + ["node_2 运行完毕"],
        cur_id=pre_id + ", node_2",
    )


# 3、定义边: 边依赖图添加内容
# 3.1 创建图，获取建造者(类似建造者模式)
builder = StateGraph(state_schema=OverAllState)
# 3.2 添加节点
builder.add_node(node_1)
builder.add_node(node_2)
# 3.3 添加边
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

# 4、获取图
graph = builder.compile()
# 5、运行图
# 需要传入初始的状态
# result = graph.invoke({"cur_id": "start"}) # 字典状态

result = graph.invoke(OverAllState(logs=[], cur_id="start"))
print(result)


{'logs': ['node_1 运行完毕', 'node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}
